In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project root
PROJECT_PATH = Path("..")

# Important project directories
DATA_PATH = PROJECT_PATH / "data"
PROCESSED_PATH = DATA_PATH / "processed"
SQL_PATH = PROJECT_PATH / "sql"
DOCS_PATH = PROJECT_PATH / "docs"
EXCEL_PATH = PROJECT_PATH / "excel"

# Create Excel directory if it doesn't exist
EXCEL_PATH.mkdir(parents=True, exist_ok=True)

print("Project path:", PROJECT_PATH.resolve())
print("Processed data:", PROCESSED_PATH.resolve())
print("Excel output:", EXCEL_PATH.resolve())

Project path: D:\fraud-loss-analysis-step1\fraud-loss-analysis
Processed data: D:\fraud-loss-analysis-step1\fraud-loss-analysis\data\processed
Excel output: D:\fraud-loss-analysis-step1\fraud-loss-analysis\excel


In [2]:
# ============================================================
# Cell 2 — Load final validation artifacts
# ============================================================

VALIDATION_PATH = PROCESSED_PATH

# Final rule validation summary
rule_validation = pd.read_csv(
    VALIDATION_PATH / "final_rule_validation_summary.csv"
)

# Rule business impact
rule_business_impact = pd.read_csv(
    VALIDATION_PATH / "final_rule_business_impact.csv"
)

# Typology-level performance
typology_performance = pd.read_csv(
    VALIDATION_PATH / "final_typology_performance.csv"
)

print("Rule validation:")
display(rule_validation)

print("\nRule business impact:")
display(rule_business_impact)

print("\nTypology performance:")
display(typology_performance)

Rule validation:


,rule_id,rule_name,role,alerts,TP,FP,FN,TN,precision_pct,recall_pct,fpr_pct,accuracy_pct,f1_pct,fraud_value_caught,fraud_value_coverage_pct,expected_fraud_prevented,review_cost,customer_friction_cost,expected_net_savings
0,R001,High-value E-commerce High-risk Category,Primary loss-prevention,616,432,184,3092,496706,70.129870,12.258797,0.037030,99.345342,20.869565,18550201.98,58.353814,1.484016e+07,92400,18400,1.472936e+07
1,R002,E-commerce Card Testing,Secondary preventive control,781,716,65,2808,496825,91.677337,20.317821,0.013081,99.425875,33.263647,18493.85,0.058177,1.479508e+04,117150,6500,-1.088549e+05



Rule business impact:


,rule,alerts,TP,FP,precision_pct,recall_pct,fraud_value_caught,expected_fraud_prevented,review_cost,customer_friction_cost,expected_net_savings
0,R001 - High-value E-commerce High-risk Category,616,432,184,70.129870,12.258797,18550201.98,1.484016e+07,92400,18400,1.472936e+07
1,R002 - E-commerce Card Testing,781,716,65,91.677337,20.317821,18493.85,1.479508e+04,117150,6500,-1.088549e+05



Typology performance:


,typology,fraud_transactions,fraud_value,R001_caught,R002_caught,R001_recall_pct,R002_recall_pct,R001_value_caught,R002_value_caught,R001_value_coverage_pct,R002_value_coverage_pct
0,account_takeover,693,21956097.34,429,0,61.904762,0.0,18457704.55,0.00,84.066418,0.0
1,card_testing,716,18493.85,0,716,0.000000,100.0,0.00,18493.85,0.000000,100.0
2,cnp_high_risk_merchant,622,1698068.01,0,0,0.000000,0.0,0.00,0.00,0.000000,0.0
3,compromised_terminal,506,2860627.61,1,0,0.197628,0.0,30320.07,0.00,1.059910,0.0
4,cross_border_spike,509,1386601.00,1,0,0.196464,0.0,32934.81,0.00,2.375219,0.0
5,impossible_travel,478,3869297.35,1,0,0.209205,0.0,29242.55,0.00,0.755759,0.0


In [3]:
# ============================================================
# Cell 3 — Business assumptions
# ============================================================

BUSINESS_ASSUMPTIONS = {
    "fraud_prevention_rate": 0.80,
    "review_cost_per_alert": 150,
    "customer_friction_per_legit_alert": 100,
    "review_capacity_per_month": 500,
}

assumptions_df = pd.DataFrame({
    "assumption": [
        "Fraud prevention rate",
        "Review cost per alert",
        "Customer friction per legitimate alert",
        "Review capacity per month",
    ],
    "value": [
        BUSINESS_ASSUMPTIONS["fraud_prevention_rate"],
        BUSINESS_ASSUMPTIONS["review_cost_per_alert"],
        BUSINESS_ASSUMPTIONS["customer_friction_per_legit_alert"],
        BUSINESS_ASSUMPTIONS["review_capacity_per_month"],
    ],
    "unit": [
        "%",
        "INR / alert",
        "INR / legitimate alert",
        "alerts / month",
    ],
})

# Display in a business-friendly format
display(assumptions_df)

,assumption,value,unit
0,Fraud prevention rate,0.8,%
1,Review cost per alert,150.0,INR / alert
2,Customer friction per legitimate alert,100.0,INR / legitimate alert
3,Review capacity per month,500.0,alerts / month


In [4]:
# ============================================================
# Cell 4 — R001 business economics
# ============================================================

r001 = rule_validation.loc[
    rule_validation["rule_id"] == "R001"
].iloc[0]

prevention_rate = BUSINESS_ASSUMPTIONS["fraud_prevention_rate"]
review_cost_per_alert = BUSINESS_ASSUMPTIONS["review_cost_per_alert"]
friction_cost_per_legit_alert = BUSINESS_ASSUMPTIONS[
    "customer_friction_per_legit_alert"
]

r001_alerts = int(r001["alerts"])
r001_tp = int(r001["TP"])
r001_fp = int(r001["FP"])
r001_fraud_value_caught = float(r001["fraud_value_caught"])

r001_expected_fraud_prevented = (
    r001_fraud_value_caught * prevention_rate
)

r001_review_cost = (
    r001_alerts * review_cost_per_alert
)

r001_friction_cost = (
    r001_fp * friction_cost_per_legit_alert
)

r001_expected_net_savings = (
    r001_expected_fraud_prevented
    - r001_review_cost
    - r001_friction_cost
)

r001_economics = pd.DataFrame({
    "metric": [
        "Alerts",
        "True positives",
        "False positives",
        "Fraud value caught",
        "Expected fraud prevented",
        "Review cost",
        "Customer friction cost",
        "Expected net savings",
    ],
    "value": [
        r001_alerts,
        r001_tp,
        r001_fp,
        r001_fraud_value_caught,
        r001_expected_fraud_prevented,
        r001_review_cost,
        r001_friction_cost,
        r001_expected_net_savings,
    ]
})

display(r001_economics)

,metric,value
0,Alerts,6.160000e+02
1,True positives,4.320000e+02
2,False positives,1.840000e+02
3,Fraud value caught,1.855020e+07
4,Expected fraud prevented,1.484016e+07
5,Review cost,9.240000e+04
6,Customer friction cost,1.840000e+04
7,Expected net savings,1.472936e+07


In [5]:
# ============================================================
# Cell 5 — R002 business economics
# ============================================================

r002 = rule_validation.loc[
    rule_validation["rule_id"] == "R002"
].iloc[0]

r002_alerts = int(r002["alerts"])
r002_tp = int(r002["TP"])
r002_fp = int(r002["FP"])
r002_fraud_value_caught = float(r002["fraud_value_caught"])

r002_expected_fraud_prevented = (
    r002_fraud_value_caught * prevention_rate
)

r002_review_cost = (
    r002_alerts * review_cost_per_alert
)

r002_friction_cost = (
    r002_fp * friction_cost_per_legit_alert
)

r002_expected_net_savings = (
    r002_expected_fraud_prevented
    - r002_review_cost
    - r002_friction_cost
)

r002_economics = pd.DataFrame({
    "metric": [
        "Alerts",
        "True positives",
        "False positives",
        "Fraud value caught",
        "Expected fraud prevented",
        "Review cost",
        "Customer friction cost",
        "Expected net savings",
    ],
    "value": [
        r002_alerts,
        r002_tp,
        r002_fp,
        r002_fraud_value_caught,
        r002_expected_fraud_prevented,
        r002_review_cost,
        r002_friction_cost,
        r002_expected_net_savings,
    ]
})

display(r002_economics)

,metric,value
0,Alerts,781.00
1,True positives,716.00
2,False positives,65.00
3,Fraud value caught,18493.85
4,Expected fraud prevented,14795.08
5,Review cost,117150.00
6,Customer friction cost,6500.00
7,Expected net savings,-108854.92


In [6]:
# ============================================================
# Cell 6 — Executive comparison of R001 vs R002
# ============================================================

comparison_df = pd.DataFrame({
    "Metric": [
        "Alerts",
        "True positives",
        "False positives",
        "Precision (%)",
        "Recall (%)",
        "Fraud value caught",
        "Expected fraud prevented",
        "Review cost",
        "Customer friction cost",
        "Expected net savings",
    ],
    "R001": [
        r001_alerts,
        r001_tp,
        r001_fp,
        float(r001["precision_pct"]),
        float(r001["recall_pct"]),
        r001_fraud_value_caught,
        r001_expected_fraud_prevented,
        r001_review_cost,
        r001_friction_cost,
        r001_expected_net_savings,
    ],
    "R002": [
        r002_alerts,
        r002_tp,
        r002_fp,
        float(r002["precision_pct"]),
        float(r002["recall_pct"]),
        r002_fraud_value_caught,
        r002_expected_fraud_prevented,
        r002_review_cost,
        r002_friction_cost,
        r002_expected_net_savings,
    ]
})

display(comparison_df)

,Metric,R001,R002
0,Alerts,6.160000e+02,781.000000
1,True positives,4.320000e+02,716.000000
2,False positives,1.840000e+02,65.000000
3,Precision (%),7.012987e+01,91.677337
4,Recall (%),1.225880e+01,20.317821
5,Fraud value caught,1.855020e+07,18493.850000
6,Expected fraud prevented,1.484016e+07,14795.080000
7,Review cost,9.240000e+04,117150.000000
8,Customer friction cost,1.840000e+04,6500.000000
9,Expected net savings,1.472936e+07,-108854.920000


In [7]:
# ============================================================
# Cell 7 — Review capacity analysis
# ============================================================

review_capacity = BUSINESS_ASSUMPTIONS["review_capacity_per_month"]

# Validation period covers Sep-Dec 2025 = 4 months
validation_months = 4

capacity_df = pd.DataFrame({
    "rule_id": ["R001", "R002"],
    "rule_name": [
        "High-value E-commerce High-risk Category",
        "E-commerce Card Testing"
    ],
    "total_alerts_validation": [
        r001_alerts,
        r002_alerts
    ],
    "validation_months": [
        validation_months,
        validation_months
    ],
    "avg_alerts_per_month": [
        r001_alerts / validation_months,
        r002_alerts / validation_months
    ],
    "monthly_capacity": [
        review_capacity,
        review_capacity
    ]
})

capacity_df["capacity_utilization_pct"] = (
    capacity_df["avg_alerts_per_month"]
    / capacity_df["monthly_capacity"]
    * 100
)

capacity_df["within_capacity"] = (
    capacity_df["avg_alerts_per_month"]
    <= capacity_df["monthly_capacity"]
)

display(capacity_df)

,rule_id,rule_name,total_alerts_validation,validation_months,avg_alerts_per_month,monthly_capacity,capacity_utilization_pct,within_capacity
0,R001,High-value E-commerce High-risk Category,616,4,154.00,500,30.80,True
1,R002,E-commerce Card Testing,781,4,195.25,500,39.05,True


In [8]:
# ============================================================
# Cell 8 — Sensitivity analysis
# ============================================================

prevention_rates = [0.50, 0.60, 0.70, 0.80, 0.90, 1.00]

sensitivity_rows = []

for rate in prevention_rates:

    # R001
    r001_prevented = r001_fraud_value_caught * rate
    r001_net = (
        r001_prevented
        - r001_review_cost
        - r001_friction_cost
    )

    # R002
    r002_prevented = r002_fraud_value_caught * rate
    r002_net = (
        r002_prevented
        - r002_review_cost
        - r002_friction_cost
    )

    sensitivity_rows.append({
        "fraud_prevention_rate_pct": rate * 100,
        "R001_expected_net_savings": r001_net,
        "R002_expected_net_savings": r002_net
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)

display(sensitivity_df)

,fraud_prevention_rate_pct,R001_expected_net_savings,R002_expected_net_savings
0,50.0,9.164301e+06,-114403.075
1,60.0,1.101932e+07,-112553.690
2,70.0,1.287434e+07,-110704.305
3,80.0,1.472936e+07,-108854.920
4,90.0,1.658438e+07,-107005.535
5,100.0,1.843940e+07,-105156.150


In [9]:
# ============================================================
# Cell 9 — Final executive recommendation
# ============================================================

recommendation_df = pd.DataFrame({
    "Decision Area": [
        "Primary production rule",
        "Secondary control",
        "R001 expected net savings",
        "R002 expected net savings",
        "R001 monthly alert volume",
        "R001 review capacity utilization",
        "R001 validation precision",
        "R001 fraud value caught",
        "Recommended action",
    ],
    "Result": [
        "R001",
        "R002",
        r001_expected_net_savings,
        r002_expected_net_savings,
        r001_alerts / validation_months,
        capacity_df.loc[
            capacity_df["rule_id"] == "R001",
            "capacity_utilization_pct"
        ].iloc[0],
        float(r001["precision_pct"]),
        r001_fraud_value_caught,
        (
            "Deploy R001 as the primary manual-review "
            "loss-prevention rule; retain R002 as a "
            "targeted card-testing preventive control "
            "rather than a manual-review rule."
        ),
    ]
})

display(recommendation_df)

,Decision Area,Result
0,Primary production rule,R001
1,Secondary control,R002
2,R001 expected net savings,14729361.584
3,R002 expected net savings,-108854.92
4,R001 monthly alert volume,154.0
5,R001 review capacity utilization,30.8
6,R001 validation precision,70.12987
7,R001 fraud value caught,18550201.98
8,Recommended action,Deploy R001 as the primary manual-review loss-...


In [10]:
# ============================================================
# Cell 10 — Prepare Excel-ready datasets
# ============================================================

# 1. Assumptions table
excel_assumptions = pd.DataFrame({
    "Assumption": [
        "Fraud prevention rate",
        "Review cost per alert",
        "Customer friction per legitimate alert",
        "Review capacity per month",
        "Validation period"
    ],
    "Value": [
        prevention_rate,
        review_cost_per_alert,
        friction_cost_per_legit_alert,
        review_capacity,
        "Sep-Dec 2025"
    ],
    "Unit": [
        "%",
        "INR / alert",
        "INR / legitimate alert",
        "alerts / month",
        "4 months"
    ]
})

# 2. Rule results
excel_rule_results = rule_validation[
    [
        "rule_id",
        "rule_name",
        "role",
        "alerts",
        "TP",
        "FP",
        "precision_pct",
        "recall_pct",
        "fraud_value_caught",
        "fraud_value_coverage_pct",
        "expected_fraud_prevented",
        "review_cost",
        "customer_friction_cost",
        "expected_net_savings"
    ]
].copy()

# 3. Capacity analysis
excel_capacity = capacity_df.copy()

# 4. Sensitivity analysis
excel_sensitivity = sensitivity_df.copy()

# 5. Typology performance
excel_typology = typology_performance.copy()

# 6. Executive summary
excel_summary = pd.DataFrame({
    "Metric": [
        "Recommended primary rule",
        "Primary rule",
        "Primary rule precision",
        "Primary rule fraud value caught",
        "Primary rule expected net savings",
        "Primary rule alerts per month",
        "Review capacity utilization",
        "Secondary control",
        "Secondary control expected net savings",
        "Business decision"
    ],
    "Result": [
        "R001",
        "E-commerce + Electronics/Gift Cards + Amount >= INR 20,000",
        f"{r001['precision_pct']:.2f}%",
        r001_fraud_value_caught,
        r001_expected_net_savings,
        r001_alerts / validation_months,
        capacity_df.loc[
            capacity_df["rule_id"] == "R001",
            "capacity_utilization_pct"
        ].iloc[0],
        "R002 - E-commerce Card Testing",
        r002_expected_net_savings,
        "Deploy R001 as the primary manual-review rule. "
        "Retain R002 as a targeted preventive control "
        "rather than a manual-review rule."
    ]
})

print("Excel datasets prepared:")
print(f"Assumptions: {excel_assumptions.shape}")
print(f"Rule results: {excel_rule_results.shape}")
print(f"Capacity: {excel_capacity.shape}")
print(f"Sensitivity: {excel_sensitivity.shape}")
print(f"Typology: {excel_typology.shape}")
print(f"Summary: {excel_summary.shape}")

display(excel_summary)

Excel datasets prepared:
Assumptions: (5, 3)
Rule results: (2, 14)
Capacity: (2, 8)
Sensitivity: (6, 3)
Typology: (6, 11)
Summary: (10, 2)


,Metric,Result
0,Recommended primary rule,R001
1,Primary rule,E-commerce + Electronics/Gift Cards + Amount >...
2,Primary rule precision,70.13%
3,Primary rule fraud value caught,18550201.98
4,Primary rule expected net savings,14729361.584
5,Primary rule alerts per month,154.0
6,Review capacity utilization,30.8
7,Secondary control,R002 - E-commerce Card Testing
8,Secondary control expected net savings,-108854.92
9,Business decision,Deploy R001 as the primary manual-review rule....


In [11]:
# ============================================================
# Cell 11 — Create Excel business-case workbook
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

excel_file = EXCEL_PATH / "fraud_rule_business_case.xlsx"

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    excel_summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    excel_assumptions.to_excel(
        writer,
        sheet_name="Assumptions",
        index=False
    )

    excel_rule_results.to_excel(
        writer,
        sheet_name="Rule Results",
        index=False
    )

    excel_capacity.to_excel(
        writer,
        sheet_name="Capacity",
        index=False
    )

    excel_sensitivity.to_excel(
        writer,
        sheet_name="Sensitivity",
        index=False
    )

    excel_typology.to_excel(
        writer,
        sheet_name="Typology Performance",
        index=False
    )


# ------------------------------------------------------------
# Apply professional formatting
# ------------------------------------------------------------

wb = load_workbook(excel_file)

header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78"
)

header_font = Font(
    color="FFFFFF",
    bold=True
)

title_font = Font(
    bold=True,
    size=14
)

thin_border = Border(
    bottom=Side(
        style="thin",
        color="D9E1F2"
    )
)

for ws in wb.worksheets:

    # Freeze header row
    ws.freeze_panes = "A2"

    # Header formatting
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )

    # Borders and alignment
    for row in ws.iter_rows():
        for cell in row:
            cell.border = thin_border
            cell.alignment = Alignment(
                vertical="center",
                wrap_text=True
            )

    # Auto-fit columns
    for column_cells in ws.columns:
        max_length = 0
        column_letter = get_column_letter(
            column_cells[0].column
        )

        for cell in column_cells:
            if cell.value is not None:
                max_length = max(
                    max_length,
                    len(str(cell.value))
                )

        ws.column_dimensions[
            column_letter
        ].width = min(max_length + 2, 45)

    # Row height
    ws.row_dimensions[1].height = 30


# ------------------------------------------------------------
# Number formatting
# ------------------------------------------------------------

# Assumptions
ws = wb["Assumptions"]

for row in range(2, ws.max_row + 1):
    if ws.cell(row, 1).value == "Fraud prevention rate":
        ws.cell(row, 2).number_format = "0%"
    elif isinstance(ws.cell(row, 2).value, (int, float)):
        ws.cell(row, 2).number_format = '#,##0.00'


# Rule Results
ws = wb["Rule Results"]

percentage_columns = [
    "precision_pct",
    "recall_pct",
    "fraud_value_coverage_pct"
]

currency_columns = [
    "fraud_value_caught",
    "expected_fraud_prevented",
    "review_cost",
    "customer_friction_cost",
    "expected_net_savings"
]

headers = {
    cell.value: cell.column
    for cell in ws[1]
}

for column_name in percentage_columns:
    col = headers.get(column_name)
    if col:
        for row in range(2, ws.max_row + 1):
            ws.cell(row, col).number_format = "0.00%"

for column_name in currency_columns:
    col = headers.get(column_name)
    if col:
        for row in range(2, ws.max_row + 1):
            ws.cell(row, col).number_format = '₹#,##0.00'


# Capacity
ws = wb["Capacity"]

headers = {
    cell.value: cell.column
    for cell in ws[1]
}

for column_name in [
    "avg_alerts_per_month",
    "monthly_capacity"
]:
    col = headers.get(column_name)
    if col:
        for row in range(2, ws.max_row + 1):
            ws.cell(row, col).number_format = "#,##0.00"

col = headers.get("capacity_utilization_pct")

if col:
    for row in range(2, ws.max_row + 1):
        ws.cell(row, col).number_format = "0.00%"


# Sensitivity
ws = wb["Sensitivity"]

headers = {
    cell.value: cell.column
    for cell in ws[1]
}

ws[headers["fraud_prevention_rate_pct"]][0].value = (
    "Fraud prevention rate (%)"
)

for row in range(2, ws.max_row + 1):
    ws.cell(
        row,
        headers["fraud_prevention_rate_pct"]
    ).number_format = "0.0"

    ws.cell(
        row,
        headers["R001_expected_net_savings"]
    ).number_format = '₹#,##0.00'

    ws.cell(
        row,
        headers["R002_expected_net_savings"]
    ).number_format = '₹#,##0.00'


# Typology Performance
ws = wb["Typology Performance"]

headers = {
    cell.value: cell.column
    for cell in ws[1]
}

for column_name in [
    "R001_recall_pct",
    "R002_recall_pct",
    "R001_value_coverage_pct",
    "R002_value_coverage_pct"
]:
    col = headers.get(column_name)

    if col:
        for row in range(2, ws.max_row + 1):
            ws.cell(row, col).number_format = "0.00%"

for column_name in [
    "fraud_value",
    "R001_value_caught",
    "R002_value_caught"
]:
    col = headers.get(column_name)

    if col:
        for row in range(2, ws.max_row + 1):
            ws.cell(row, col).number_format = '₹#,##0.00'


# Summary formatting
ws = wb["Summary"]

for row in range(2, ws.max_row + 1):

    metric = ws.cell(row, 1).value

    if metric in [
        "Primary rule fraud value caught",
        "Primary rule expected net savings",
        "Secondary control expected net savings"
    ]:
        ws.cell(row, 2).number_format = '₹#,##0.00'

    elif metric == "Primary rule precision":
        # Already stored as text such as 70.13%
        pass

    elif metric in [
        "Primary rule alerts per month",
        "Review capacity utilization"
    ]:
        ws.cell(row, 2).number_format = "#,##0.00"


# Save workbook
wb.save(excel_file)

print("Excel workbook created successfully.")
print("File:", excel_file.resolve())
print("Size:", round(excel_file.stat().st_size / 1024, 2), "KB")

Excel workbook created successfully.
File: D:\fraud-loss-analysis-step1\fraud-loss-analysis\excel\fraud_rule_business_case.xlsx
Size: 10.29 KB


In [12]:
# ============================================================
# Cell 12 — Excel workbook QA
# ============================================================

from openpyxl import load_workbook

wb_check = load_workbook(
    excel_file,
    data_only=False
)

expected_sheets = [
    "Summary",
    "Assumptions",
    "Rule Results",
    "Capacity",
    "Sensitivity",
    "Typology Performance"
]

print("Workbook:", excel_file.name)
print("\nSheets found:")
print(wb_check.sheetnames)

print("\nSheet dimensions:")

for sheet_name in expected_sheets:
    ws = wb_check[sheet_name]
    print(
        f"{sheet_name}: "
        f"{ws.max_row} rows × {ws.max_column} columns"
    )

print("\nSheet validation:")

missing_sheets = [
    sheet for sheet in expected_sheets
    if sheet not in wb_check.sheetnames
]

if missing_sheets:
    print("ERROR — Missing sheets:", missing_sheets)
else:
    print("All expected sheets are present.")

# ------------------------------------------------------------
# Verify key business values
# ------------------------------------------------------------

ws = wb_check["Rule Results"]

headers = {
    cell.value: cell.column
    for cell in ws[1]
}

rule_ids = [
    ws.cell(row, headers["rule_id"]).value
    for row in range(2, ws.max_row + 1)
]

print("\nRule IDs found:", rule_ids)

for row in range(2, ws.max_row + 1):

    rule_id = ws.cell(
        row,
        headers["rule_id"]
    ).value

    if rule_id == "R001":
        print("\nR001 verification:")
        print(
            "Precision:",
            ws.cell(row, headers["precision_pct"]).value
        )
        print(
            "Fraud value caught:",
            ws.cell(row, headers["fraud_value_caught"]).value
        )
        print(
            "Expected net savings:",
            ws.cell(row, headers["expected_net_savings"]).value
        )

    elif rule_id == "R002":
        print("\nR002 verification:")
        print(
            "Precision:",
            ws.cell(row, headers["precision_pct"]).value
        )
        print(
            "Fraud value caught:",
            ws.cell(row, headers["fraud_value_caught"]).value
        )
        print(
            "Expected net savings:",
            ws.cell(row, headers["expected_net_savings"]).value
        )

print("\nWorkbook QA completed.")

Workbook: fraud_rule_business_case.xlsx

Sheets found:
['Summary', 'Assumptions', 'Rule Results', 'Capacity', 'Sensitivity', 'Typology Performance']

Sheet dimensions:
Summary: 11 rows × 2 columns
Assumptions: 6 rows × 3 columns
Rule Results: 3 rows × 14 columns
Capacity: 3 rows × 8 columns
Sensitivity: 7 rows × 3 columns
Typology Performance: 7 rows × 11 columns

Sheet validation:
All expected sheets are present.

Rule IDs found: ['R001', 'R002']

R001 verification:
Precision: 70.12987012987013
Fraud value caught: 18550201.98
Expected net savings: 14729361.584

R002 verification:
Precision: 91.67733674775928
Fraud value caught: 18493.85
Expected net savings: -108854.92

Workbook QA completed.
